In [1]:
import time
import numpy as np
import primme
from qiskit.transpiler import CouplingMap
import fulqrum as fq

In [2]:
# Build 1541-qubit coupling map
cmap = CouplingMap.from_heavy_square(23)
num_qubits = cmap.size()

# Generate Hamiltonian
H = fq.QubitOperator(num_qubits, [])
touched_edges = set({})
coeffs = [1/2, 1/2, 1]
for edge in cmap.get_edges():
    if edge[::-1] not in touched_edges:
        H += fq.QubitOperator(num_qubits, [("XX", edge, coeffs[0]), 
                                           ("YY", edge, coeffs[1]), 
                                           ("ZZ", edge, coeffs[2])])
        touched_edges.add(edge)

# 1 million Pseudo counts
counts = {}
for kk in range(int(1e7)):
    counts[bin(kk)[2:].zfill(num_qubits)] = 1

# Solve eigenproblem (can substitute scipy.sparse.linalg.eigsh)
S = fq.Subspace([list(counts)])

In [3]:
Hsub = fq.SubspaceHamiltonian(H, S)

In [8]:
from scipy.sparse.linalg import LinearOperator
matvec_times = []       # list of (cpu_s, gpu_s) per matvec call
_prev_matvec_end = [None]

def timed_matvec(x):
    cpu_time = (
        time.perf_counter() - _prev_matvec_end[0]
        if _prev_matvec_end[0] is not None
        else 0.0
    )
    t0 = time.perf_counter()
    result = Hsub.matvec(x)
    t1 = time.perf_counter()
    gpu_time = t1 - t0
    print(f"  matvec #{len(matvec_times) + 1:4d} | Matvec: {gpu_time * 1000:8.2f} ms | Eigen: {cpu_time * 1000:8.2f} ms")
    _prev_matvec_end[0] = t1
    matvec_times.append((cpu_time, gpu_time))
    return result

H_timed = LinearOperator(Hsub.shape, matvec=timed_matvec, dtype=Hsub.dtype)

In [9]:
evals, _ = primme.eigsh(H_timed, k=1, which='SA', method='PRIMME_DEFAULT_MIN_MATVECS', tol=1e-3)

  matvec #   1 | Matvec:  1171.72 ms | Eigen:     0.00 ms
  matvec #   2 | Matvec:  1112.23 ms | Eigen:   162.94 ms
  matvec #   3 | Matvec:  1109.38 ms | Eigen:   193.42 ms
  matvec #   4 | Matvec:  1107.28 ms | Eigen:   210.09 ms
  matvec #   5 | Matvec:  1108.19 ms | Eigen:   233.83 ms
  matvec #   6 | Matvec:  1107.80 ms | Eigen:   259.56 ms
  matvec #   7 | Matvec:  1095.32 ms | Eigen:   549.36 ms
  matvec #   8 | Matvec:  1097.70 ms | Eigen:   375.59 ms
  matvec #   9 | Matvec:  1125.27 ms | Eigen:   411.12 ms
  matvec #  10 | Matvec:  1102.73 ms | Eigen:   435.90 ms
  matvec #  11 | Matvec:  1099.23 ms | Eigen:   467.46 ms
  matvec #  12 | Matvec:  1100.58 ms | Eigen:   499.49 ms
  matvec #  13 | Matvec:  1098.56 ms | Eigen:   522.77 ms
  matvec #  14 | Matvec:  1107.36 ms | Eigen:   551.59 ms
  matvec #  15 | Matvec:  1098.21 ms | Eigen:   590.74 ms
  matvec #  16 | Matvec:  1117.57 ms | Eigen:   957.63 ms
  matvec #  17 | Matvec:  1102.89 ms | Eigen:   366.16 ms
  matvec #  18

In [6]:
assert np.allclose(evals[0], 1934.20344492)

In [7]:
np.mean([item[1] for item in matvec_times[1:]])

np.float64(1.101509530277124)